# Phase 10: Advanced Training Techniques

## What you'll learn
- Mixed precision training (AMP)
- Gradient accumulation
- Distributed training (DDP)
- DataParallel vs DistributedDataParallel
- Multi-GPU strategies
- Profiling with torch.profiler

---

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 10.1 Mixed Precision Training (AMP)

Uses **float16** for most operations (faster, less memory) and **float32** where precision matters.

Benefits:
- ~2x faster training on modern GPUs
- ~50% less GPU memory
- Minimal accuracy impact

In [ ]:
model = nn.Sequential(
    nn.Linear(784, 256), nn.ReLU(),
    nn.Linear(256, 10)
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# AMP components
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

# Dummy data
X = torch.randn(1000, 784).to(device)
y = torch.randint(0, 10, (1000,)).to(device)
loader = DataLoader(TensorDataset(X, y), batch_size=64)

# Training with AMP
model.train()
for inputs, labels in loader:
    optimizer.zero_grad()

    # Forward pass in mixed precision
    with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
        outputs = model(inputs)
        loss = criterion(outputs, labels)

    # Backward with gradient scaling
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

print(f"AMP training complete. Loss: {loss.item():.4f}")

## 10.2 Gradient Accumulation

Simulate larger batch sizes when GPU memory is limited.

Instead of one big batch of 256, use 4 mini-batches of 64 and accumulate gradients.

In [ ]:
model = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10)).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

accumulation_steps = 4  # effective batch = 64 * 4 = 256
loader = DataLoader(TensorDataset(X, y), batch_size=64)

model.train()
optimizer.zero_grad()

for step, (inputs, labels) in enumerate(loader):
    outputs = model(inputs)
    loss = criterion(outputs, labels) / accumulation_steps  # scale loss
    loss.backward()  # gradients accumulate

    if (step + 1) % accumulation_steps == 0:
        optimizer.step()   # update weights
        optimizer.zero_grad()  # reset gradients

print(f"Gradient accumulation complete")

## 10.3 DataParallel vs DistributedDataParallel

| Feature | DataParallel | DistributedDataParallel |
|---------|-------------|------------------------|
| Processes | Single | Multiple |
| GIL bottleneck | Yes | No |
| Speed | Slower | Faster |
| Recommended | Quick prototyping | Production training |

In [ ]:
# --- DataParallel (simple but slower) ---
# Wraps model to split batches across GPUs
if torch.cuda.device_count() > 1:
    model_dp = nn.DataParallel(model)
    print(f"DataParallel on {torch.cuda.device_count()} GPUs")
else:
    print(f"Single GPU/CPU — DataParallel not needed")
    print(f"Available GPUs: {torch.cuda.device_count()}")

In [ ]:
# --- DistributedDataParallel (recommended) ---
# This must be run as a script, not in a notebook
# Save this pattern for your training scripts:

ddp_script = '''
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler

def setup(rank, world_size):
    dist.init_process_group("nccl", rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)

def train(rank, world_size):
    setup(rank, world_size)

    model = YourModel().to(rank)
    model = DDP(model, device_ids=[rank])

    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank)
    loader = DataLoader(dataset, batch_size=64, sampler=sampler)

    for epoch in range(epochs):
        sampler.set_epoch(epoch)  # important for shuffling
        for inputs, labels in loader:
            # standard training loop
            pass

    dist.destroy_process_group()

# Launch: torchrun --nproc_per_node=4 train.py
'''
print(ddp_script)

## 10.4 Profiling with torch.profiler

Find bottlenecks in your training loop.

In [ ]:
model = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10)).to(device)
optimizer = optim.AdamW(model.parameters())
criterion = nn.CrossEntropyLoss()

with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU],
    record_shapes=True
) as prof:
    for i in range(5):
        inputs = torch.randn(64, 784).to(device)
        labels = torch.randint(0, 10, (64,)).to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=10))

## 10.5 Memory Optimization Tips

Practical techniques to reduce GPU memory usage.

In [ ]:
# 1. Gradient checkpointing — trade compute for memory
from torch.utils.checkpoint import checkpoint

class MemoryEfficientModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(nn.Linear(784, 512), nn.ReLU())
        self.block2 = nn.Sequential(nn.Linear(512, 256), nn.ReLU())
        self.head = nn.Linear(256, 10)

    def forward(self, x):
        # Recompute block1/block2 during backward instead of storing activations
        x = checkpoint(self.block1, x, use_reentrant=False)
        x = checkpoint(self.block2, x, use_reentrant=False)
        return self.head(x)

# 2. Clear cache
if torch.cuda.is_available():
    print(f"Memory allocated: {torch.cuda.memory_allocated() / 1e6:.1f} MB")
    torch.cuda.empty_cache()
    print(f"After cache clear: {torch.cuda.memory_allocated() / 1e6:.1f} MB")

# 3. Delete intermediate tensors
# del large_tensor
# torch.cuda.empty_cache()

print("Memory optimization techniques demonstrated")

## ✅ Phase 10 Summary

You now know:
- Mixed precision (AMP) for 2x speedup
- Gradient accumulation for larger effective batches
- DDP for multi-GPU training
- Profiling to find bottlenecks
- Memory optimization techniques

**Next → Phase 11: Generative Models**